#### Enviroment Setup

In [ ]:
%pip install -r requirements.txt

#### Loading dataset and filtering it

In [ ]:
import pandas as pd
from bertopic import BERTopic

In [ ]:
#Reading csv
df = pd.read_csv(r'dataset\datasetA.csv')

In [ ]:
#Selecting just the title, link, and classes_str columns.
keyData = df[['title', 'classes_str', 'link']]
#Checking nulls
keyData.isna().sum()

In [ ]:
#Filtering the df so it only selects rows where the classes_str value contains 'Learning, Knowledge & Education'
df2 = keyData[keyData['classes_str'].str.contains('Learning, Knowledge & Education', case=False)]
df2.head()

#### Topic Modeling

In [ ]:
#BERTopic only reads lists so i did this
docs = df2["title"].tolist()
print(len(docs))

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

#Topic is the topic number. -1 refers to outliers so it can be ignored.
#Count is how many rows from title fit that topic
#Name is the topic name
#Representation is the top words that summarize the topic
#Representative_Docs is the example documents that best illustrate the topic

In [ ]:
docDf = topic_model.get_document_info(docs)
docDf.head()

In [ ]:
linksTitles = keyData[['link', 'title']]
merged = pd.merge(docDf, linksTitles, how='inner', left_on='Document', right_on='title')
merged = merged.drop(columns=["title"])
merged = merged.rename(columns={"Document": "Title"})
print(merged.shape)
merged.head()

In [ ]:
topic1_df = merged[merged['Topic'] == 0]
print(topic1_df.shape)
topic1_df = topic1_df.reset_index(drop=True)
topic1_df.head()


#### Scraper Setup

In [ ]:
from newspaper import Article
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import nltk
nltk.download('punkt_tab')

In [ ]:
# log file for Emmanuel's debuging ventures
import logging
logging.basicConfig(filename='scraperLog.log', format='%(levelname)s: %(asctime)s: %(message)s', datefmt='%m/%d/%Y - %I:%M:%S %p', level=logging.DEBUG)
logging.getLogger('selenium').setLevel(logging.WARNING)
log = logging.getLogger(__name__)
#levels: debug, info, warning, error, critical

In [ ]:
# parses the javascript of the html to redirect and retrieve the correct url
def retrieve_url(driver, link: str, row_index: int = -1, wait_time: int = 1.5) -> (str | None):
    url = None

    try:
        log.debug('POSTing')
        driver.get(link)
        log.debug('POST request completed.')

        later = time.time() + wait_time
        while time.time() < later:
            time.sleep(0.01)
        log.debug('Wait time completed.')

        url = driver.execute_cdp_cmd("Runtime.evaluate", {"expression":"location.href"})['result']['value']
        driver.execute_script("window.stop()")
        log.debug('Chrome JS scripts executed.')

        if url.startswith('https://news.google.com') and wait_time < 2:
            log.info(f'Retrying url for at row {row_index} for 3 seconds.')
            url = retrieve_url(driver, link, row_index, 3)

    except Exception as e:
            log.info(f"Couldn't retrieve url at row {row_index}.") if row_index > -1 else log.info("Couldn't retrieve url.")
            log.error(e)

    return url

In [ ]:
# return dictionary of authors, keywords, summary, and text
def scrape(url: str, row_index: int = -1) -> (dict[str | list[str]] | None):
    try:
        article = Article(url)
        article.build()
        text = article.text

        if not text:
            row = f' for row {row_index}' if row_index > -1 else ''
            raise Exception(f'Newspaper3k did not return any text{row}.')

        return {'authors': article.authors,
                'keywords': article.keywords,
                'summary': article.summary,
                'text': text}
    except Exception as e:
        log.info(f"Failed to scrape article at row {row_index}.")
        log.error(e)
        return

#returns only the text
def scrape_text(url: str, row_index: int = -1) -> (str):
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if not text:
            row = f' for row {row_index}' if row_index > -1 else ''
            raise Exception(f'Newspaper3k did not return any text{row}.')
        return text
    except Exception as e:
        log.info(f"Failed to scrape article at row {row_index}.")
        log.error(e)
        return "Failed to scrape"


In [ ]:
#chrome options setup for headless browser and high efficiency
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-renderer-backgrounding")
chrome_options.add_argument("--disable-background-timer-throttling")
chrome_options.add_argument("--disable-backgrounding-occluded-windows")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-cache")
chrome_options.add_argument("--disk-cache-size=0")
chrome_options.add_argument("--media-cache-size=0")
chrome_options.add_argument("--disable-site-isolation-trials")
chrome_options.add_argument("--disable-background-networking")
chrome_options.add_argument("--disable-features=TranslateUI,BackForwardCache")
chrome_options.add_argument("--disable-sync")
chrome_options.page_load_strategy = "none"

In [ ]:
# input the information and class that you want to scrape, will create a new updated csv
def create_scraped_dataset(dataframe, saveFilepath: str, filter_column: str = 'text', filter_value: str = '', BATCH_SIZE: int = 100):
    newDataframe = dataframe
    newDataframe['text'] = ''
    NUM_ROWS = len(newDataframe)

    with webdriver.Chrome(chrome_options) as driver:
        for row in range(NUM_ROWS):
            ENTRY_IS_BLANK = (not bool(newDataframe['text'][row]))

            if filter_value.lower() in newDataframe[filter_column][row].lower() and ENTRY_IS_BLANK:
                link = newDataframe['link'][row]
                url = retrieve_url(driver, link, row)

                if url == None:
                    log.info(f'URL is None trying to reset driver at row {row}.')
                    driver.quit()
                    driver = webdriver.Chrome(chrome_options)
                    log.info('Driver reset.')
                    url = retrieve_url(driver, link, row)

                #url still equal to 'None' after second attempt - pack it up, ggs
                if url == None:
                    log.info('URL still None.')
                    data = "Selenium did not handle redirect"
                else:
                    data = scrape_text(url, row)

                newDataframe['text'][row] = data

            log.debug(f'Row {row} completed.')

            #Process in batches of 100
            if (row + 1) % BATCH_SIZE == 0:
                newDataframe.to_csv(saveFilepath, index=False)
                driver.quit()
                driver = webdriver.Chrome(chrome_options)
                log.info(f'{row + 1} rows completed, file saved, driver restarted.')

    newDataframe.to_csv(saveFilepath, index=False)
    log.info("Full table walkthrough completed.")

    return newDataframe

#### Scraper

In [ ]:
# Arguments are the dataframe that is passed in, output filepath for saving data,
# filter_column / filter_value: optional arguments;
# in case only want to scrape row where df['filter_colum'][row] contains filter_value,
# batch_size: optional; 100 by default; not important but could reduce bugs if lowered.

# This method will return a dataframe with scraped data and create a file
try:
    topic1_df = create_scraped_dataset(topic1_df, 'dataset/datasetA_class1_scrape_topic1.csv')
except Exception as e:
    log.error(e)

In [ ]:
topic1_df